# Machine Learning Models Implementation

This module implements elastic net regression and boosting(LightGBM) models. Compared to linear regression. Return average MSE from cross validation.
Since the ytm data has many outliers, we also use classification models to predict the direction of movement.

In [1]:
import pandas as pd

data = pd.read_parquet('../data/final_merged_clean.parquet', engine='pyarrow')

data.sort_values(by=['bond_cusip', 'date'], inplace=True)

In [2]:
macro_columns = ["sp500_ret", "ir3m_chg", "ir10y_chg",
    "vix_chg", "gdp_gr", "cpi_infl"] # Macro feature columns which need normalization
data = data.drop(columns=['gs3m', 'term_spread']) # Duplicates of ir3m and ir10y

data.loc[:, 'ytm_chg'] = data.groupby(['bond_cusip'])['ytm'].diff()

data.dropna(subset=['ytm_chg'], inplace=True)

data['up_down'] = data['ytm_chg'].apply(lambda x: 'up' if x > 1e-6 else ('down' if x < -1e-6 else 'neutral'))


In [3]:
# Set parameters
train_year = 10
val_year = 1
test_year = 1
random_seed = 666

start_year = data['date'].min().year
end_year = data['date'].max().year - test_year - val_year - train_year

In [4]:
idx_cols = ['bond_cusip', 'date']
y_col_reg = 'ytm_chg'
y_col_clf = 'up_down'
x_cols = [col for col in data.columns if col not in idx_cols + [y_col_reg, y_col_clf, 'ytm']]  # Exclude target and index columns from features


In [5]:
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, accuracy_score, precision_score, recall_score

from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMRegressor, LGBMClassifier


In [6]:
def normalize(train_data: pd.DataFrame,
              test_data: pd.DataFrame,
              cols: list) -> pd.DataFrame:
    scaler = StandardScaler()
    scaler.fit(train_data[cols])
    train_scaled= scaler.transform(train_data[cols])
    test_scaled = scaler.transform(test_data[cols])
    train_data.loc[:, cols] = train_scaled
    test_data.loc[:, cols] = test_scaled
    return train_data, test_data

In [7]:

def calc_metrics(y_pred, y_test, task_type: str):
    if task_type == 'regression':
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        medae = median_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        return mse, mae, medae, r2
    elif task_type == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        return accuracy, precision, recall
    else:
        raise ValueError("Unsupported task type")

In [8]:
def one_load(model: any, 
             model_name: str,
             data: pd.DataFrame,
             x_cols: list,
             y_col: str,
             idx_cols: list,
             year_: int,
             train_year: int,
             val_year: int,
             test_year: int,
             task_type: str='regression'
             ) -> pd.DataFrame:
    

    data_train = data[data['date'].dt.year <= (year_ + train_year + val_year)]
    data_test = data[(data['date'].dt.year > (year_ + train_year + val_year)) & 
                    (data['date'].dt.year <= (year_ + train_year + val_year + test_year))]

    data_train, data_test = normalize(data_train, data_test, macro_columns)
    X_train = data_train[x_cols]
    y_train = data_train[y_col]
    X_test = data_test[x_cols]
    y_test = data_test[y_col]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if task_type == 'regression':

        mse, mae, medae, r2 = calc_metrics(y_pred, y_test, task_type)

        df = pd.DataFrame({
            'start_date': [data_test['date'].min()],
            'end_date': [data_test['date'].max()],
            'model': [model_name],
            'mse': [mse],
            'mae': [mae],
            'medae': [medae],
            'r2': [r2]
        })
    else:
        accuracy, precision, recall_score = calc_metrics(y_pred, y_test, task_type)

        df = pd.DataFrame({
            'start_date': [data_test['date'].min()],
            'end_date': [data_test['date'].max()],
            'model': [model_name],
            'accuracy': [accuracy],
            'precision': [precision],
            'recall': [recall_score]    
        })

    df_result = data_test[idx_cols].copy()
    df_result.loc[:, 'model'] = model_name
    df_result.loc[:, 'pred'] = y_pred
    df_result.loc[:, 'act'] = y_test.values
    return df, df_result


In [9]:

def main(model: any, 
         model_name: str,
         y_col: str,
         x_cols: list,
         idx_cols: list,
         data: pd.DataFrame,
         start_year: int,
         end_year: int,
         train_year: int,
         val_year: int,
         test_year: int,
         task_type: str='regression'
         ) -> pd.DataFrame:

    df_metrics = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
    df_results = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])

    for year_ in range(start_year, end_year + 1):
        print(f"Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
        df, df_result = one_load(model, model_name, data, x_cols, y_col, idx_cols, year_, train_year, val_year, test_year, task_type)
        df_metrics = pd.concat([df_metrics, df], ignore_index=True)
        df_results = pd.concat([df_results, df_result], ignore_index=True)
    return df_metrics, df_results
    


## Linear Regression

In [10]:
# linear regression
lm = LinearRegression()
metrics_lm, results_lm = main(model=lm, 
         model_name='Linear Regression',
         y_col=y_col_reg,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='regression')


Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018
Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


## Elastic Net Regression

In [11]:
data_train = data[data['date'].dt.year <= (start_year + train_year + val_year)]
X_train = data_train[x_cols]
y_train = data_train[y_col_reg]
# Tuning Alpha and L1 ratio with cross-validation once and use for all years as tuning per year is computationally expensive
elastic_net_cv = ElasticNetCV(l1_ratio=[0, 0.01, 0.05, .1, .5, .7, .9, .95, .99, 1], 
                              alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0], 
                              cv=10,
                              random_state=random_seed,
                              n_jobs=-1)
elastic_net_cv.fit(X_train, y_train)
best_alpha = elastic_net_cv.alpha_
best_l1_ratio = elastic_net_cv.l1_ratio_
print(f"Best alpha: {best_alpha}, Best l1_ratio: {best_l1_ratio}")

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
/opt/homebrew/Caskroom/minic

Best alpha: 0.001, Best l1_ratio: 0.0


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.132e+01, tolerance: 1.250e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


In [12]:
# Elastic Net Regression
elastic_net = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio, random_state=random_seed)
df_metrics_enet, df_results_enet = main(model=elastic_net, 
         model_name='Elastic Net Regression',
         y_col=y_col_reg,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='regression')

Forecasting for period starting 2013 to 2014


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.111e+01, tolerance: 1.250e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/va

Forecasting for period starting 2014 to 2015


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.298e+01, tolerance: 1.288e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Forecasting for period starting 2015 to 2016


...

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.295e+01, tolerance: 1.705e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Forecasting for period starting 2022 to 2023


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.482e+01, tolerance: 1.749e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


Forecasting for period starting 2023 to 2024


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.603e+01, tolerance: 1.777e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


## Boosting Regression (LightGBM)

In [13]:
max_leaves = [7, 15, 31, 63, 127]
num_trees = [100, 200, 500]
learning_speed = [0.01, 0.05, 0.1, 0.2]

best_params = {'num_leaves': None, 'n_estimators': None, 'learning_rate': None}
best_mse = float('inf')
for leaves in max_leaves:
    for trees in num_trees:
        for speed in learning_speed:
            print(f"Training LGBM with leaves={leaves}, trees={trees}, learning_rate={speed}")
            lgbm = LGBMRegressor(max_leaves=leaves, n_estimators=trees, learning_rate=speed, random_state=random_seed, n_jobs=-1)
            df_metrics_lgbm, df_results_lgbm = main(model=lgbm, 
                     model_name=f'LGBM Reg (leaves={leaves}, trees={trees}, lr={speed})',
                     y_col=y_col_reg,
                     x_cols=x_cols,
                     idx_cols=idx_cols,
                     data=data,
                     start_year=start_year,
                     end_year=end_year,
                     train_year=train_year,
                     val_year=val_year,
                     test_year=test_year,
                     task_type='regression')
            avg_mse = df_metrics_lgbm['mse'].mean()
            if avg_mse < best_mse:
                best_mse = avg_mse
                best_params['num_leaves'] = leaves
                best_params['n_estimators'] = trees
                best_params['learning_rate'] = speed


Training LGBM with leaves=7, trees=100, learning_rate=0.01
Forecasting for period starting 2013 to 2014
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006628 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9430
[LightGBM] [Info] Number of data points in the train set: 323307, number of used features: 53
[LightGBM] [Info] Start training from score -0.000305
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.000329
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008051 seconds.
You can set `force_row_wise=true` t

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007706 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.000329
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007003 seconds.
You can set `force_row_wise=true` t

...

[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007166 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.000329
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007150 seconds.
You can set `force_row_wise

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006751 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.000329
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006827 seconds.
You can set `force_row_wise

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006940 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.000329
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007145 seconds.
You can set `force_row_wise

In [14]:
print(best_params)

{'num_leaves': 7, 'n_estimators': 500, 'learning_rate': 0.01}


In [15]:
lgbm = LGBMRegressor(max_leaves=best_params['num_leaves'],
                      n_estimators=best_params['n_estimators'],
                      learning_rate=best_params['learning_rate'],
                      random_state=random_seed, n_jobs=-1)
metrics_lgbm, results_lgbm = main(model=lgbm, 
        model_name='LGBM Regressor',
         y_col=y_col_reg,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='regression')


Forecasting for period starting 2013 to 2014
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009960 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9430
[LightGBM] [Info] Number of data points in the train set: 323307, number of used features: 53
[LightGBM] [Info] Start training from score -0.000305
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006766 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.000329
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006874 seconds.
You can set `force_row_wise=true` t

## Ensemble

In [16]:
from sklearn.ensemble import StackingRegressor

elastic_net = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio, random_state=random_seed)
lgbm = LGBMRegressor(max_leaves=best_params['num_leaves'],
                      n_estimators=best_params['n_estimators'],
                      learning_rate=best_params['learning_rate'],
                      random_state=random_seed, n_jobs=-1)
base_models = [
    ('elastic_net', elastic_net),
    ('lgbm', lgbm)
]
meta_model = LinearRegression()
stacked_model = StackingRegressor(estimators=base_models, final_estimator=meta_model, n_jobs=-1)
metrics_stacked, results_stacked = main(model=stacked_model, 
        model_name='Stacked Regressor',
         y_col=y_col_reg,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='regression')

Forecasting for period starting 2013 to 2014
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9430
[LightGBM] [Info] Number of data points in the train set: 323307, number of used features: 53
[LightGBM] [Info] Start training from score -0.000305


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.111e+01, tolerance: 1.250e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing 

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.130e+01, tolerance: 8.479e-03
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.646e+01, tolerance: 9.498e-03
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one o

[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2014 to 2015


...

[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2023 to 2024
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015679 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10003
[LightGBM] [Info] Number of data points in the train set: 839463, number of used features: 53
[LightGBM] [Info] Start training from score 0.000005


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.603e+01, tolerance: 1.777e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current v

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.119e+01, tolerance: 1.470e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.065e+01, tolerance: 1.256e-02
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one o

[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31


## Linear Classification

In [17]:
lm_clf = LogisticRegression(max_iter=1000, random_state=random_seed, n_jobs=-1)
metrics_lm_clf, results_lm_clf = main(model=lm_clf, 
         model_name='Logistic Regression',
         y_col=y_col_clf,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='classification')


Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018
Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


## Elastic Net Classification

In [18]:
# Tuning Alpha and L1 ratio with cross-validation once and use for all years as tuning per year is computationally expensive
data_train = data[data['date'].dt.year <= (start_year + train_year + val_year)]
X_train = data_train[x_cols]
y_train = data_train[y_col_clf]

l1_ratios_clf = [0, 0.01, 0.05, .1, .5, .7, .9, .95, .99, 1]
alphas_clf = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

best_alpha_clf = None
best_l1_ratio_clf = None
best_accuracy = 0
for l1_ratio in l1_ratios_clf: 
    for alpha in alphas_clf:
        print(f"Training Elastic Net Classifier with alpha={alpha}, l1_ratio={l1_ratio}")
        enet_clf = LogisticRegression(penalty='elasticnet', 
                                      solver='saga', 
                                      l1_ratio=l1_ratio, 
                                      C=1/alpha, 
                                      max_iter=1000,
                                      random_state=random_seed, n_jobs=-1)
        enet_clf.fit(X_train, y_train)
        y_pred = enet_clf.predict(X_train)
        accuracy = accuracy_score(y_train, y_pred)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_alpha_clf = alpha
            best_l1_ratio_clf = l1_ratio
print(f"Best alpha: {best_alpha_clf}, Best l1_ratio: {best_l1_ratio_clf}")

Training Elastic Net Classifier with alpha=0.001, l1_ratio=0
Training Elastic Net Classifier with alpha=0.01, l1_ratio=0
Training Elastic Net Classifier with alpha=0.1, l1_ratio=0
Training Elastic Net Classifier with alpha=1.0, l1_ratio=0
Training Elastic Net Classifier with alpha=10.0, l1_ratio=0
Training Elastic Net Classifier with alpha=100.0, l1_ratio=0
Training Elastic Net Classifier with alpha=0.001, l1_ratio=0.01
Training Elastic Net Classifier with alpha=0.01, l1_ratio=0.01
Training Elastic Net Classifier with alpha=0.1, l1_ratio=0.01
Training Elastic Net Classifier with alpha=1.0, l1_ratio=0.01
Training Elastic Net Classifier with alpha=10.0, l1_ratio=0.01
Training Elastic Net Classifier with alpha=100.0, l1_ratio=0.01
Training Elastic Net Classifier with alpha=0.001, l1_ratio=0.05
Training Elastic Net Classifier with alpha=0.01, l1_ratio=0.05
Training Elastic Net Classifier with alpha=0.1, l1_ratio=0.05
Training Elastic Net Classifier with alpha=1.0, l1_ratio=0.05
Training El

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Elastic Net Classifier with alpha=1.0, l1_ratio=0.95
Training Elastic Net Classifier with alpha=10.0, l1_ratio=0.95
Training Elastic Net Classifier with alpha=100.0, l1_ratio=0.95
Training Elastic Net Classifier with alpha=0.001, l1_ratio=0.99
Training Elastic Net Classifier with alpha=0.01, l1_ratio=0.99
Training Elastic Net Classifier with alpha=0.1, l1_ratio=0.99


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Elastic Net Classifier with alpha=1.0, l1_ratio=0.99
Training Elastic Net Classifier with alpha=10.0, l1_ratio=0.99
Training Elastic Net Classifier with alpha=100.0, l1_ratio=0.99
Training Elastic Net Classifier with alpha=0.001, l1_ratio=1
Training Elastic Net Classifier with alpha=0.01, l1_ratio=1
Training Elastic Net Classifier with alpha=0.1, l1_ratio=1


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Elastic Net Classifier with alpha=1.0, l1_ratio=1


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Training Elastic Net Classifier with alpha=10.0, l1_ratio=1
Training Elastic Net Classifier with alpha=100.0, l1_ratio=1
Best alpha: 1.0, Best l1_ratio: 0.05


In [19]:
# Elastic Net Classification
elastic_net_clf = LogisticRegression(penalty='elasticnet', 
                                     solver='saga', 
                                     l1_ratio=best_l1_ratio_clf, 
                                     C=1/best_alpha_clf, 
                                     max_iter=1000,
                                     random_state=random_seed,
                                     n_jobs=-1)
metrics_enet_clf, results_enet_clf = main(model=elastic_net_clf, 
         model_name='Elastic Net Classification',
         y_col=y_col_clf,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='classification')

Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018
Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


## Boosting Classification (LightGBM)

In [20]:
# Tuning hyperparameters

max_leaves = [7, 15, 31, 63, 127]
num_trees = [100, 200, 500]
learning_speed = [0.01, 0.05, 0.1, 0.2]

best_params_clf = {'num_leaves': None, 'n_estimators': None, 'learning_rate': None}
best_accuracy_clf = 0
for leaves in max_leaves:
    for trees in num_trees:
        for speed in learning_speed:
            print(f"Training LGBM Classifier with leaves={leaves}, trees={trees}, learning_rate={speed}")
            lgbm_clf = LGBMClassifier(max_leaves=leaves, n_estimators=trees, learning_rate=speed, random_state=random_seed, n_jobs=-1)
            df_metrics_lgbm_clf, df_results_lgbm_clf = main(model=lgbm_clf, 
                     model_name=f'LGBM Clf (leaves={leaves}, trees={trees}, lr={speed})',
                     y_col=y_col_clf,
                     x_cols=x_cols,
                     idx_cols=idx_cols,
                     data=data,
                     start_year=start_year,
                     end_year=end_year,
                     train_year=train_year,
                     val_year=val_year,
                     test_year=test_year,
                     task_type='classification')
            avg_accuracy = df_metrics_lgbm_clf['accuracy'].mean()
            if avg_accuracy > best_accuracy_clf:
                best_accuracy_clf = avg_accuracy
                best_params_clf['num_leaves'] = leaves
                best_params_clf['n_estimators'] = trees
                best_params_clf['learning_rate'] = speed
print(best_params_clf)


Training LGBM Classifier with leaves=7, trees=100, learning_rate=0.01
Forecasting for period starting 2013 to 2014
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007467 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9430
[LightGBM] [Info] Number of data points in the train set: 323307, number of used features: 53
[LightGBM] [Info] Start training from score -0.615257
[LightGBM] [Info] Start training from score -3.614360
[LightGBM] [Info] Start training from score -0.838025
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2014 to 2015


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006481 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007307 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves

...

Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006575 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006737 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007499 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=127 will be ignored. Current

In [22]:
lgbm_clf = LGBMClassifier(max_leaves=best_params_clf['num_leaves'],
                          n_estimators=best_params_clf['n_estimators'],
                          learning_rate=best_params_clf['learning_rate'],
                          random_state=random_seed, n_jobs=-1)
metrics_lgbm_clf, results_lgbm_clf = main(model=lgbm_clf, 
         model_name='LGBM Classifier',
         y_col=y_col_clf,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='classification')


Forecasting for period starting 2013 to 2014
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007085 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9430
[LightGBM] [Info] Number of data points in the train set: 323307, number of used features: 53
[LightGBM] [Info] Start training from score -0.615257
[LightGBM] [Info] Start training from score -3.614360
[LightGBM] [Info] Start training from score -0.838025
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006752 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
Forecasting for period starting 2015 to 2016
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: 

In [23]:
from sklearn.ensemble import StackingClassifier

base_models_clf = [
    ('logistic_regression', lm_clf),
    ('elastic_net', elastic_net_clf),
    ('lgbm', lgbm_clf)
]
meta_model_clf = LogisticRegression(max_iter=1000, random_state=random_seed, n_jobs=-1)
stacked_model_clf = StackingClassifier(estimators=base_models_clf, final_estimator=meta_model_clf, n_jobs=-1)
metrics_stacked_clf, results_stacked_clf = main(model=stacked_model_clf, 
        model_name='Stacked Classifier',
         y_col=y_col_clf,
         x_cols=x_cols,
         idx_cols=idx_cols,
         data=data,
         start_year=start_year,
         end_year=end_year,
         train_year=train_year,
         val_year=val_year,
         test_year=test_year,
         task_type='classification')

Forecasting for period starting 2013 to 2014
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007239 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9430
[LightGBM] [Info] Number of data points in the train set: 323307, number of used features: 53
[LightGBM] [Info] Start training from score -0.615257
[LightGBM] [Info] Start training from score -3.614360
[LightGBM] [Info] Start training from score -0.838025
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_metrics = pd.concat([df_metrics, df], ignore_index=True)
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_35191/1161305422.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008374 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9495
[LightGBM] [Info] Number of data points in the train set: 364018, number of used features: 53
[LightGBM] [Info] Start training from score -0.608209
[LightGBM] [Info] Start training from score -3.595118
[LightGBM] [Info] Start training from score -0.848123
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] num_leaves is set=31, max_leaves=7 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing 

In [31]:
df_results_clf = pd.concat([results_lm_clf, results_enet_clf, results_lgbm_clf, results_stacked_clf], ignore_index=True)
df_metrics_clf = pd.concat([metrics_lm_clf, metrics_enet_clf, metrics_lgbm_clf, metrics_stacked_clf], ignore_index=True)

In [25]:

df_results_reg = pd.concat([results_lm, df_results_enet, results_lgbm, results_stacked], ignore_index=True)
df_metrics_reg = pd.concat([metrics_lm, df_metrics_enet, metrics_lgbm, metrics_stacked], ignore_index=True)

In [32]:
ave_reg = df_metrics_reg.groupby('model')[['mse', 'mae', 'medae', 'r2']].mean().reset_index()
ave_clf = df_metrics_clf.groupby('model')[['accuracy', 'precision', 'recall']].mean().reset_index()
print(ave_reg)
print(ave_clf)

                    model       mse       mae     medae        r2
0  Elastic Net Regression  0.000093  0.003167  0.002032  0.005956
1          LGBM Regressor  0.000084  0.002817  0.001607  0.085135
2       Linear Regression  0.000093  0.003171  0.002036  0.004982
3       Stacked Regressor  0.000084  0.002841  0.001607  0.063947
                        model  accuracy  precision    recall
0  Elastic Net Classification  0.682760   0.694127  0.682760
1             LGBM Classifier  0.713273   0.722421  0.713273
2         Logistic Regression  0.682736   0.694145  0.682736
3          Stacked Classifier  0.706753   0.718828  0.706753
